# 06 — Ensemble residual + LightGBM no ph (EF01), regime anual (treino 2024)
Piso sazonal-naive + 288 LGBM no resíduo + DLinear-res + NNLS (pesos na val, 4 fatias). Régua 02 recarregada. 2025 intocado.

In [1]:
import json
import pickle
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_ph_2024.csv"
OUT = ROOT / "univariavel" / "resultados" / "06-ensemble-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24")]
LN = 2016
LGB_EST, LGB_LR, LGB_LEAVES = 150, 0.05, 31
LGB_STRIDE = 2
DL_EPOCHS, DL_PAT = 30, 5
ENS_STRIDE = 4
TR_INF_STRIDE = 16
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 11665 (11.1%)


,ds,y
count,105121,93456.00000
mean,2024-07-01 12:00:00,5.90729
min,2024-01-01 00:00:00,5.21000
25%,2024-04-01 06:00:00,5.71000
50%,2024-07-01 12:00:00,5.92000
75%,2024-09-30 18:00:00,6.09000
max,2024-12-31 00:00:00,6.68000
std,NaN,0.30356


## 2. EDA

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("ph EF01 2024 — série completa (treino)")
ax[0].set_ylabel("ph")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 4971 passos = 414.2 h | gaps > 24 passos: 9


fig salva


## 3. Limpeza

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 6408
  outage 2024-01-16 11:35:00 → 2024-01-18 17:50:00 (652 slots)
  outage 2024-02-13 17:00:00 → 2024-02-13 17:05:00 (2 slots)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots)
  outage 2024-04-29 11:30:00 → 2024-05-02 03:10:00 (765 slots)
  outage 2024-05-27 12:00:00 → 2024-06-13 16:10:00 (4947 slots)
  outage 2024-10-19 06:05:00 → 2024-10-19 06:30:00 (6 slots)
  outage 2024-10-19 13:35:00 → 2024-10-19 13:50:00 (4 slots)
  outage 2024-11-25 15:15:00 → 2024-11-25 15:20:00 (2 slots)
  outage 2024-12-03 13:30:00 → 2024-12-03 14:50:00 (17 slots)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-1.79 p-valor=0.384 → NÃO estacionária


fig salva


## 5. Janelamento + val 4 fatias

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy().astype(np.float32)  # float32: corta a cópia das janelas pela metade (métricas a 4 casas intactas)
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date
is_val = np.zeros(len(ends), dtype=bool)
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")
va = np.where(is_val)[0]
tr = np.where(~is_val)[0]
print(f"treino: {len(tr)} janelas | val: {len(va)} janelas | descartadas (NaN): {len(s)-L-H+1-len(X)}")
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
daily_idx = np.where((ends.time == pd.Timestamp("23:55").time()) & is_val)[0]
print("dias-âncora na val:", len(daily_idx))

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
treino: 24659 janelas | val: 10080 janelas | descartadas (NaN): 61455
dias-âncora na val: 35


## 6. Métricas + baselines de referência

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino:", pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val:", pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())

treino:                       MAE    RMSE    MAPE   sMAPE
persistencia       0.0740  0.1040  1.2896  1.2913
sazonal_naive_288  0.0627  0.0937  1.0927  1.0961
media_movel_288    0.0668  0.0965  1.1616  1.1651
val:                       MAE    RMSE    MAPE   sMAPE
persistencia       0.0564  0.0809  0.9779  0.9774
sazonal_naive_288  0.0421  0.0614  0.7325  0.7320
media_movel_288    0.0468  0.0636  0.8100  0.8103


## 7. LightGBM no resíduo (288 modelos, um por passo)

In [8]:
import lightgbm as lgb

def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

tr2 = tr[::LGB_STRIDE]
Xb_tr = X[tr2]
Ftr, emtr = base_feats(Xb_tr, ends[tr2])
Str = np.stack([Xb_tr[:, L - SEASON + h] for h in range(H)], axis=1)
Rtr = (Y[tr2] - Str).astype(np.float32)
print(f"features: {Ftr.shape} + hora do passo | resíduo std: {Rtr.std():.4f}")

models = []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    m = lgb.LGBMRegressor(n_estimators=LGB_EST, learning_rate=LGB_LR, num_leaves=LGB_LEAVES,
                          verbosity=-1, force_col_wise=True)
    m.fit(np.column_stack([Ftr, sh, ch]), Rtr[:, j])
    models.append(m)
    if (j + 1) % 72 == 0:
        print(f"  lgbm {j+1}/{H} ...", flush=True)
print(f"lgbm: {len(models)} modelos em {time.time()-t0:.0f}s")
with open(OUT / "modelos" / "lgbm_steps.pkl", "wb") as f:
    pickle.dump(models, f)
print("modelos salvos: modelos/lgbm_steps.pkl (gitignored, >100MB?)")

def prevê_lgbm(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]
    F, em = base_feats(Xb, ends[ii])
    S = np.stack([Xb[:, L - SEASON + h] for h in range(H)], axis=1)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, m in enumerate(models):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] + m.predict(np.column_stack([F, sh, ch]))
    return P

imp = np.mean([m.booster_.feature_importance(importance_type="gain") for m in models], axis=0)
nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "hora_sin", "hora_cos"]
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM — importância média das features (gain, 288 modelos)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:5]])
print("fig salva: 07-importancia-lgbm.png")
del Ftr, Rtr, Str, Xb_tr

features: (12330, 25) + hora do passo | resíduo std: 0.0937


  lgbm 72/288 ...


  lgbm 144/288 ...


  lgbm 216/288 ...


  lgbm 288/288 ...


lgbm: 288 modelos em 28s


modelos salvos: modelos/lgbm_steps.pkl (gitignored, >100MB?)
top features: [('rs288', 195.7), ('rm2016', 194.2), ('rs144', 79.3), ('rm288', 79.3), ('rs36', 33.4)]
fig salva: 07-importancia-lgbm.png


## 8. DLinear-res + régua 02 + NNLS na val

In [9]:
def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

Xr_tr, Rr_tr = monta_res(tr[::LGB_STRIDE])
Xr_va, Rr_va = monta_res(va[::ENS_STRIDE])
print(f"residual: treino {Xr_tr.shape} val {Xr_va.shape}")
dlres = DLinearLite().to(DEVICE)
opt = torch.optim.Adam(dlres.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_tr), torch.from_numpy(Rr_tr)), batch_size=512, shuffle=True)
va_loader = DataLoader(TensorDataset(torch.from_numpy(Xr_va), torch.from_numpy(Rr_va)), batch_size=512)
best, patience = float("inf"), 0
t0 = time.time()
for ep in range(1, DL_EPOCHS + 1):
    dlres.train()
    for xb, yb in tr_loader:
        opt.zero_grad(); loss = loss_fn(dlres(xb), yb); loss.backward(); opt.step()
    dlres.eval(); vl = 0.0
    with torch.no_grad():
        for xb, yb in va_loader:
            vl += float(loss_fn(dlres(xb), yb)) * len(xb)
    vl /= len(va_loader.dataset)
    tag = ""
    if vl < best:
        best, patience = vl, 0
        torch.save({"state": dlres.state_dict()}, OUT / "modelos" / "dlinear_res_ph.pt")
        tag = " *"
    else:
        patience += 1
    print(f"dlres ep {ep:02d} val={vl:.5f}{tag}", flush=True)
    if patience >= DL_PAT:
        break
print(f"dlres em {time.time()-t0:.0f}s | melhor val={best:.5f}")
dlres.load_state_dict(torch.load(OUT / "modelos" / "dlinear_res_ph.pt", map_location="cpu", weights_only=False)["state"])
dlres.eval()
del Xr_tr, Rr_tr, Xr_va, Rr_va

class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

CKPT02P = ROOT / "univariavel" / "resultados" / "02-lstnet-ph" / "modelos" / "lstnet_ph.pt"
assert CKPT02P.exists(), "rode o 02-lstnet-ph antes! (" + str(CKPT02P) + " ausente)"
ckpt02 = torch.load(CKPT02P, map_location="cpu", weights_only=False)
ruler = LSTNet1D().to(DEVICE)
ruler.load_state_dict(ckpt02["state"])
ruler.eval()
print("régua 02 recarregada:", CKPT02P)
val5 = s.to_numpy().astype(np.float32)
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Wln = sliding_window_view(val5, LN)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), LN, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1

@torch.no_grad()
def prevê_tudo(idxs, batch=256):
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    Gb = prevê_lgbm(ii)
    outs = []
    Xt = torch.from_numpy(X[ii][:, -2016:].astype(np.float32))
    for b in range(0, len(Xt), batch):
        outs.append(dlres(Xt[b:b+batch]).numpy())
    Dr = Ps + np.concatenate(outs)
    return Ps, Gb, Dr

@torch.no_grad()
def prevê_ruler(idxs, batch=256):
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(ruler(xb, tb).numpy())
    return np.concatenate(outs)

from scipy.optimize import nnls
va2 = va[::ENS_STRIDE]
Ps_v, Gb_v, Dr_v = prevê_tudo(va2)
Pn_v = prevê_ruler(va2)
A = np.column_stack([Ps_v.ravel(), Pn_v.ravel(), Gb_v.ravel(), Dr_v.ravel()])
w, _ = nnls(A, Y[va2].ravel())
pesos = {k: round(float(v), 4) for k, v in zip(["sazonal", "lstnet", "lgbm", "dlres"], w)}
json.dump({"pesos": pesos, "mode": "nnls-ensemble sobre sazonal+lstnet+lgbm+dlres", "val_slices": VAL_SLICES},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "lgbm-direct-288 + dlinear-residual + nnls-ensemble", "LN": LN},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("pesos ensemble (nnls na val):", pesos)

def ensemble(Ps, Pn, Gb, Dr):
    return w[0]*Ps + w[1]*Pn + w[2]*Gb + w[3]*Dr

t0 = time.time()
tr3 = tr[::TR_INF_STRIDE]
Ps_tr, Gb_tr, Dr_tr = prevê_tudo(tr3); Pn_tr = prevê_ruler(tr3)
En_tr = ensemble(Ps_tr, Pn_tr, Gb_tr, Dr_tr)
Ps_v2, Gb_v2, Dr_v2 = prevê_tudo(va); Pn_v2 = prevê_ruler(va)
En_va = ensemble(Ps_v2, Pn_v2, Gb_v2, Dr_v2)
Ps_d, Gb_d, Dr_d = prevê_tudo(daily_idx); Pn_d = prevê_ruler(daily_idx)
En_d = ensemble(Ps_d, Pn_d, Gb_d, Dr_d)
print(f"inferência em {time.time()-t0:.0f}s")
print("ENS treino:", {k: round(v, 4) for k, v in metricas(Y[tr3], En_tr).items()})
print("ENS val:", {k: round(v, 4) for k, v in metricas(Yva, En_va).items()})

residual: treino (12330, 2016) val (2520, 2016)


dlres ep 01 val=0.00409 *


dlres ep 02 val=0.00383 *


dlres ep 03 val=0.00379 *


dlres ep 04 val=0.00373 *


...

dlres ep 10 val=0.00373


dlres ep 11 val=0.00373


dlres em 10s | melhor val=0.00362
régua 02 recarregada: /home/marcos/temporal-model/resultados/02-lstnet-ph/modelos/lstnet_ph.pt


pesos ensemble (nnls na val): {'sazonal': 0.2701, 'lstnet': 0.7247, 'lgbm': 0.0, 'dlres': 0.0045}


inferência em 11s
ENS treino: {'MAE': 0.0491, 'RMSE': 0.0737, 'MAPE': 0.8558, 'sMAPE': 0.8581}
ENS val: {'MAE': 0.0357, 'RMSE': 0.0513, 'MAPE': 0.6203, 'sMAPE': 0.6204}


## 9. Tabelas

In [10]:
linhas = {m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}
linhas["lstnet(02)"] = metricas(Yva, Pn_v2)
linhas["lgbm"] = metricas(Yva, Gb_v2)
linhas["dlres"] = metricas(Yva, Dr_v2)
linhas["ens"] = metricas(Yva, En_va)
tab_va = pd.DataFrame(linhas).T.round(4)
tab_va.to_csv(OUT / "metricas_val.csv")
print("=== val ===")
print(tab_va.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]}
diario["lstnet(02)"] = metricas(Yd, Pn_d)
diario["lgbm"] = metricas(Yd, Gb_d)
diario["dlres"] = metricas(Yd, Dr_d)
diario["ens"] = metricas(Yd, En_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_val_diaria.csv")
print("=== val dias-âncora ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstnet(02)"] = [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))]
por_dia["lgbm"] = [mae(Yd[k:k+1], Gb_d[k:k+1]) for k in range(len(Yd))]
por_dia["dlres"] = [mae(Yd[k:k+1], Dr_d[k:k+1]) for k in range(len(Yd))]
por_dia["ens"] = [mae(Yd[k:k+1], En_d[k:k+1]) for k in range(len(Yd))]
por_dia.to_csv(OUT / "metricas_por_dia.csv")
print(por_dia.round(4).to_string())
print(f"\nMelhor na val: {tab_va['MAE'].idxmin()} = {tab_va['MAE'].min():.4f}")
print(f"pesos ensemble: {pesos}")

=== val ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0564  0.0809  0.9779  0.9774
sazonal_naive_288  0.0421  0.0614  0.7325  0.7320
media_movel_288    0.0468  0.0636  0.8100  0.8103
lstnet(02)         0.0373  0.0529  0.6479  0.6475
lgbm               0.0501  0.0830  0.8787  0.8728
dlres              0.0421  0.0592  0.7314  0.7311
ens                0.0357  0.0513  0.6203  0.6204
=== val dias-âncora ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0638  0.0892  1.1154  1.1057
sazonal_naive_288  0.0421  0.0614  0.7322  0.7317
media_movel_288    0.0458  0.0619  0.7921  0.7925
lstnet(02)         0.0360  0.0514  0.6300  0.6290
lgbm               0.0465  0.0752  0.8145  0.8102
dlres              0.0411  0.0572  0.7166  0.7158
ens                0.0340  0.0495  0.5944  0.5941
            persistencia  sazonal_naive_288  media_movel_288  lstnet(02)    lgbm   dlres     ens
2024-04-19        0.0305             0.0262           0.0277      0

## 10. Figuras

In [11]:
ks = [0, len(tr3) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp = cheap_preds(X[tr3])
for ax, k, j in zip(axes, ks, range(3)):
    tf = pd.date_range(ends[tr3[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr3[k]], freq="5min")
    ax.plot(tf, Y[tr3[k]], "k-", lw=1.5, label="real")
    ax.plot(tf, cp["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pn_tr[k], lw=1, alpha=0.6, label="lstnet(02)")
    ax.plot(tf, Gb_tr[k], lw=1, alpha=0.9, label="lgbm")
    ax.plot(tf, En_tr[k], lw=1.2, alpha=0.9, label="ens")
    ax.set_title(f"origem {ends[tr3[k]]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab_va["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE na val — todos os modelos (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, ax = plt.subplots(figsize=(12, 3.5))
for col, ls in [("sazonal_naive_288", "--"), ("lstnet(02)", "-."), ("ens", "-"), ("lgbm", ":"), ("persistencia", ":")]:
    if col in por_dia.columns:
        ax.plot(pd.to_datetime(por_dia.index), por_dia[col], ls, lw=1.1, label=col)
ax.set_title("ph — MAE por dia-âncora na val")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")
print("figs salvas")

figs salvas


## 11. Conclusões
Checkpoints em `modelos/` para o benchmark 2025 (08).